In [0]:
from pyspark.sql import functions as F

# Read October file
df_oct = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("/Volumes/workspace/ecommerce/ecommerce_data/2019-Oct.csv")

# Read November file
df_nov = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("/Volumes/workspace/ecommerce/ecommerce_data/2019-Nov.csv")

In [0]:
events = df_oct.unionByName(df_nov)

In [0]:
events.printSchema()
events.count()

In [0]:
events.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("bronze_events")

In [0]:
events = spark.table("bronze_events")

In [0]:
# Silver Layer Formtion

features_df = events_clean.groupBy("user_id").agg(
    F.count("*").alias("total_events"),
    F.count(F.when(F.col("event_type")=="purchase",1)).alias("total_purchases"),
    F.sum("price").alias("total_spent"),
    F.avg("price").alias("avg_price"),
    F.max("event_time").alias("last_activity")
)

In [0]:
features_df.select("user_id").distinct().count() == features_df.count()

In [0]:
features_df.describe().show()

In [0]:
spark.sql("DROP TABLE IF EXISTS silver_user_features")

In [0]:
features_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_user_features")